# Detección de enfermedades en hojas de plantas con transfer learning

Proyecto final de Machine Learning (UTEC). Clasificación de hojas sanas vs. enfermas con el dataset PlantVillage y ResNet50 preentrenada en ImageNet.

Ejecutar en Google Colab con GPU (Entorno de ejecución > Cambiar tipo de entorno > GPU).

## 1. Configuración

In [ ]:
!pip install -q datasets scikit-learn

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

MODO = "binario"      # "binario" (sana vs. enferma) o "multiclase" (38 clases)
IMG_SIZE = 224
BATCH = 32
FRACCION = 1.0        # bajar a 0.3 si el entrenamiento tarda demasiado
EPOCAS_FASE1 = 5
EPOCAS_FASE2 = 5

## 2. Carga del dataset

Usamos la partición oficial 80/20, que agrupa las imágenes por hoja para evitar fuga de información entre train y test.

In [ ]:
ds = load_dataset("mohanty/PlantVillage", "color")
print(ds)
print(ds["train"].features)

In [ ]:
feat = ds["train"].features["label"]
def nombre_label(v):
    return feat.int2str(v) if hasattr(feat, "int2str") else str(v)

clases_38 = sorted({nombre_label(v) for v in ds["train"]["label"]})
idx_38 = {n: i for i, n in enumerate(clases_38)}

def objetivo(v):
    n = nombre_label(v)
    if MODO == "binario":
        return 0 if "healthy" in n.lower() else 1   # 0 = sana, 1 = enferma
    return idx_38[n]

CLASES = ["sana", "enferma"] if MODO == "binario" else clases_38
N_CLASES = len(CLASES)
print(N_CLASES, "clases")

## 3. Partición de validación por hoja

Separamos un 15% del train como validación usando `leaf_id`, para que una misma hoja no aparezca en train y validación.

In [ ]:
train_full = ds["train"]
if FRACCION < 1.0:
    train_full = train_full.shuffle(seed=SEED).select(range(int(len(train_full) * FRACCION)))
test_hf = ds["test"]
if FRACCION < 1.0:
    test_hf = test_hf.shuffle(seed=SEED).select(range(int(len(test_hf) * FRACCION)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, va_idx = next(gss.split(np.zeros(len(train_full)), groups=train_full["leaf_id"]))
train_hf, val_hf = train_full.select(tr_idx), train_full.select(va_idx)

y_train = np.array([objetivo(v) for v in train_hf["label"]])
y_val = np.array([objetivo(v) for v in val_hf["label"]])
y_test = np.array([objetivo(v) for v in test_hf["label"]])
print(len(train_hf), len(val_hf), len(test_hf))

## 4. EDA: distribución de clases y ejemplos

In [ ]:
valores, conteos = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4) if MODO == "multiclase" else (5, 4))
plt.bar([CLASES[v] for v in valores], conteos)
plt.xticks(rotation=90 if MODO == "multiclase" else 0)
plt.title("Distribución de clases en train")
plt.tight_layout(); plt.savefig("distribucion_clases.png", dpi=150); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.ravel(), np.random.choice(len(train_hf), 10, replace=False)):
    ax.imshow(train_hf[int(i)]["image"]); ax.set_title(nombre_label(train_hf[int(i)]["label"])[:25], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig("ejemplos.png", dpi=150); plt.show()

## 5. Pipeline: reescalado, normalización ImageNet y data augmentation

In [ ]:
def a_tf(hf, y, entrenamiento):
    def gen():
        for ej, etiqueta in zip(hf, y):
            img = ej["image"].convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            yield np.asarray(img, dtype=np.uint8), etiqueta
    d = tf.data.Dataset.from_generator(
        gen, output_signature=(tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.uint8),
                               tf.TensorSpec((), tf.int64)))
    if entrenamiento:
        d = d.shuffle(2048, seed=SEED)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = a_tf(train_hf, y_train, True)
val_ds = a_tf(val_hf, y_val, False)
test_ds = a_tf(test_hf, y_test, False)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name="augmentation")

## 6. Modelo: ResNet50 preentrenada + cabeza de clasificación

In [ ]:
base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

entrada = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(entrada)
x = tf.keras.applications.resnet50.preprocess_input(tf.cast(x, tf.float32))
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
reg = tf.keras.regularizers.l2(1e-4)
if MODO == "binario":
    salida = tf.keras.layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)(x)
    perdida, metricas = "binary_crossentropy", ["accuracy", tf.keras.metrics.AUC(name="auc"),
                                               tf.keras.metrics.Recall(name="recall"),
                                               tf.keras.metrics.Precision(name="precision")]
else:
    salida = tf.keras.layers.Dense(N_CLASES, activation="softmax", kernel_regularizer=reg)(x)
    perdida, metricas = "sparse_categorical_crossentropy", ["accuracy"]
modelo = tf.keras.Model(entrada, salida)
modelo.summary()

## 7. Entrenamiento en dos fases con pesos de clase y early stopping

In [ ]:
pesos = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(pesos))
print("Pesos de clase:", class_weight)

callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
             tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2)]

# Fase 1: base congelada, solo se entrena la cabeza
modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=perdida, metrics=metricas)
h1 = modelo.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE1,
                class_weight=class_weight, callbacks=callbacks)

In [ ]:
# Fase 2: fine-tuning del último bloque convolucional (conv5) con learning rate bajo
base.trainable = True
for capa in base.layers:
    capa.trainable = capa.name.startswith("conv5") and not isinstance(capa, tf.keras.layers.BatchNormalization)

modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=perdida, metrics=metricas)
h2 = modelo.fit(train_ds, validation_data=val_ds, epochs=EPOCAS_FASE2,
                class_weight=class_weight, callbacks=callbacks)
modelo.save("resnet50_plantvillage.keras")

In [ ]:
def unir(h1, h2, k): return h1.history[k] + h2.history[k]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, k in enumerate(["loss", "accuracy"]):
    ax[i].plot(unir(h1, h2, k), label="train"); ax[i].plot(unir(h1, h2, "val_" + k), label="val")
    ax[i].axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray"); ax[i].set_title(k); ax[i].legend()
plt.tight_layout(); plt.savefig("curvas_entrenamiento.png", dpi=150); plt.show()

## 8. Evaluación en test

In [ ]:
prob = modelo.predict(test_ds)
if MODO == "binario":
    prob = prob.ravel(); y_pred = (prob >= 0.5).astype(int)
    auc = roc_auc_score(y_test, prob)
else:
    y_pred = prob.argmax(1)
    auc = roc_auc_score(y_test, prob, multi_class="ovr", average="macro")

print(classification_report(y_test, y_pred, target_names=CLASES, digits=4))
print("AUC:", round(auc, 4))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=CLASES if MODO == "binario" else None).plot(cmap="Greens")
plt.title("Matriz de confusión (test)"); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

if MODO == "binario":
    RocCurveDisplay.from_predictions(y_test, prob); plt.title(f"Curva ROC (AUC = {auc:.4f})")
    plt.savefig("curva_roc.png", dpi=150); plt.show()

## 9. Análisis de errores

Ejemplos mal clasificados para la sección de Discusión.

In [ ]:
errores = np.where(y_pred != y_test)[0]
print(len(errores), "errores de", len(y_test))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.ravel(), errores[:10]):
    ax.imshow(test_hf[int(i)]["image"])
    ax.set_title(f"real: {CLASES[y_test[i]][:15]}\npred: {CLASES[y_pred[i]][:15]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig("errores.png", dpi=150); plt.show()